# 04 - SVM method

**Aim:** implement the practical SVM protocol for signal-peptide detection, with feature extraction and parameter selection inside nested cross-validation.

We used the practical instructions in `04 - Methods - SVM.pdf` to plan this section. We will reuse the representative proteins and folds from 02b, build composition and hydrophobicity features, and select the SVM settings using training data only. The analysis in 02c provides biological context, and 03 provides our loading and reporting conventions.

**Materials**
- `04 - Methods - SVM.pdf` (slides 22–26 and 30–41; practical instructions on slide 41).
- `04_feature_extraction_hydrophobicity.ipynb` (setup, A.1–A.5 and B.4).
- `04_SVM-sklearn_revised.ipynb` (sections 1–3 and 7).
- `03_von_heijne_method.ipynb` (loading in 1.1; metrics and mean ± SE in 5.4–5.6).

**Status:** implementation is in progress; remaining tasks are marked pending.

## Work distribution and handoffs

We defined five consecutive blocks so each person can complete their contribution and pass the cumulative notebook to the next. Person 1 handles shared preparation; Person 5 handles final integration. Everyone can read the materials in advance, but implementation follows **Person 1 → Person 2 → Person 3 → Person 4 → Person 5**.

| Person | Scientific sections | Receives | Passes on |
| --- | --- | --- | --- |
| 1 | 0–1: data and common definitions | Prepared FASTA/TSV and practical materials. | To Person 2: `data`, reserved `benchmarking_data`, paths, fixed hydropathy settings and the training-only `get_n` helper. |
| 2 | 2: feature extraction | Person 1's data and definitions. | To Person 3: composition/profile functions, `extract_features(proteins, n)`, feature names and catalog. |
| 3 | 3: RF ranking and SVM helpers | Person 2's extractors and feature order. | To Person 4: ranking, coupled parameter-grid, scaler/SVM and evaluation helpers. |
| 4 | 4–5: inner selection and outer evaluation | Person 3's helpers and all preceding contributions. | To Person 5: five external-fold metrics, predictions, selected settings/features and inner-search scores. |
| 5 | 6: reporting and final integration | Person 4's results and the cumulative notebook. | To the group: integrated notebook, result tables, figures and observations. |

We kept names and interfaces common across the blocks:

| Object | Interface |
| --- | --- |
| `data` | Training proteins with `accession`, `sequence`, `label`, `split`, `cv_fold` and original metadata. |
| `get_n(train_data)` | Integer N-terminal length derived only from positive SP lengths in the supplied training subset. |
| `extract_features(proteins, n)` | Numeric DataFrame indexed by accession, with the same ordered feature columns for every subset. |
| RF ranking | Column indices ordered by importance, plus names and importances from the current training fit. |
| Search candidates | Valid tuples `(k, C, gamma)`, with gamma calculated for that candidate's k. |
| `inner_scores` | `test_fold`, `validation_fold`, `n`, `k`, `C`, `gamma`, `mcc`. |
| `cv_per_round` | `test_fold`, `n`, `selected_k`, `C`, `gamma`, `precision`, `recall`, `accuracy`, `mcc`, `f1`. |
| `cv_predictions` | `accession`, `test_fold`, `label`, `prediction`. |
| `selected_features` | `test_fold`, `feature`, `rank`, `importance`, after the outer-training refit. |

All fitted quantities belong to their current training subset. The handoffs contain definitions and reusable functions first, then actual results after Person 4 completes CV. Each person adds the explanation for their own block before handing it on.

## 0. Data loading

We kept paths relative to `notebooks/` and selected the Conda `LB2` environment and the same kernel as 03.

### 0.1 Imports and paths

**Implemented — Person 1.**

We prepared the libraries for data handling, plotting, RF selection, scaling, SVM classification and evaluation, following the imports in the SVM example. We kept relative paths from `notebooks/`, as in 03. We created the folders for this section's tables and figures.

The Kyte–Doolittle scale is imported in 1.2 using Biopython, listed in `requirements.txt`.

**Similar code:** imports in `04_SVM-sklearn_revised.ipynb`, section 0; relative paths and plot settings in 03 section 0.

In [11]:
# Imports follow section 0 in 04_SVM-sklearn_revised.ipynb.
# Path handling follows section 0 in 03_von_heijne_method.ipynb.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

# These paths and plot settings follow section 0 in 03_von_heijne_method.ipynb.
input_dir = Path("../data/prepared")
table_dir = Path("../results/tables/04")
figure_dir = Path("../results/figures/04")

table_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme()


### 0.2 Load the prepared proteins

**Implemented — Person 1.**

We reused the FASTA reader and accession mapping from 03. The TSV files provide the metadata, and the matching FASTA files provide the sequences. We attached each sequence by accession and assigned label 1 to SP and 0 to NO_SP. The loaded tables contain 8,146 training proteins and 2,038 benchmarking proteins.

We kept the split and fold assignments from 02b. Cross-validation will use `data`, containing the training proteins; `benchmarking_data` will remain separate. Fold identifiers stay 1–5, with missing fold values allowed for benchmarking.

**Similar code:** 03 section 1.1 supplies this loader. The SVM example section 1 uses a random split of another dataset; we retained our prepared partitions instead.

In [12]:
# This reader is reused from section 1.1 in 03_von_heijne_method.ipynb.
def read_fasta(fasta_path):
    sequences = {}
    with open(fasta_path, encoding="utf-8") as fasta_in:
        for line in fasta_in:
            line = line.strip()
            if line.startswith(">"):
                accession = line[1:].split()[0]
                sequences[accession] = ""
            elif line:
                sequences[accession] += line
    return sequences


# TSV/FASTA loading and accession mapping also follow section 1.1 in that notebook.
positive_data = pd.read_csv(
    input_dir / "positive_dataset.tsv", sep="\t", dtype={"cv_fold": "Int64"}
)
negative_data = pd.read_csv(
    input_dir / "negative_dataset.tsv", sep="\t", dtype={"cv_fold": "Int64"}
)
positive_sequences = read_fasta(input_dir / "positive_dataset.fasta")
negative_sequences = read_fasta(input_dir / "negative_dataset.fasta")

positive_data["sequence"] = positive_data["accession"].map(positive_sequences)
negative_data["sequence"] = negative_data["accession"].map(negative_sequences)
positive_data["label"] = 1
negative_data["label"] = 0

# Keep the saved partitions, as in 03 section 1.1; no random split from the SVM example.
prepared_data = pd.concat([positive_data, negative_data], ignore_index=True)
data = prepared_data[prepared_data["split"] == "training"].copy()
benchmarking_data = prepared_data[prepared_data["split"] == "benchmarking"].copy()

print("Training proteins:", len(data))
print("Benchmarking proteins:", len(benchmarking_data))


Training proteins: 8146
Benchmarking proteins: 2038


## 1. Common feature definitions

We separated fixed choices from quantities learned in each training subset. The composition length n will be learned from positive training proteins; it is not a fixed 90-residue window (slides 30 and 36–40).

### 1.1 Estimate n from the current training subset

**Implemented — Person 1.**

We defined `get_n(train_data)` to estimate the composition length from the positive proteins in the supplied training subset (slides 30 and 40). Their `cleavage_pos` gives the last SP residue and therefore the annotated SP length.

We chose to round a half-integer median upward so n can be used for sequence slicing. This is our convention; the slides do not specify a rounding rule.

The function will be called separately for each three-fold inner-training set and again for the four-fold outer-training refit. The resulting n will also be used for that split's validation or test proteins. `cleavage_pos` contributes only to this training estimate, not to the predictor matrix.

**Similar code:** neither practical notebook includes this helper. Slide 40 specifies the calculation, and 02c establishes the meaning of `cleavage_pos`.

In [13]:
# This helper implements the training-only median in slides 30 and 40 of
# 04 - Methods - SVM.pdf; neither practical example contains an equivalent function.
def get_n(train_data):
    sp_lengths = train_data.loc[train_data["label"] == 1, "cleavage_pos"]
    median_length = sp_lengths.median()
    # Rounding upward is our project convention, not a rule from the examples.
    return int(np.ceil(median_length))


### 1.2 Fixed hydropathy settings and residue policy

**Implemented — Person 1.**

We used the Kyte–Doolittle scale supplied by Biopython, as in the hydrophobicity example. Positive values indicate hydrophobic residues and negative values indicate hydrophilic residues. We chose the first 40 residues and windows of 5, following the starting example in slides 32–34. These settings will stay fixed during the k/C/gamma search; composition will use its separately estimated n.

We chose complete windows with uniform weights and no padding. For unknown residues, Person 2's extractor will preserve their positions and average the known values within each original window. A window with no known values will be missing, and the profile summaries will use the non-missing values. Composition will exclude non-canonical residues from its denominator. Shorter sequences will use the available N-terminal region.

We kept the canonical amino-acid alphabet in the same order as the planned composition columns. The scale and settings below complete the shared definitions for Person 2; profile extraction belongs to section 2.

**Similar code:** hydrophobicity notebook setup and A.1–A.3 supply the scale and complete-window example. A.5 explains known-residue averaging, but its final `hydrophobicity_profile` shrinks windows at the ends; Person 2 will adapt that logic to our complete-window choice.

In [14]:
# Scale lookup follows setup and A.1 in 04_feature_extraction_hydrophobicity.ipynb.
from Bio.SeqUtils.ProtParamData import kd

hydropathy_scale = kd
# Shared alphabet for our composition extractor; not supplied by the hydropathy example.
aa_alphabet = list("ACDEFGHIKLMNPQRSTVWY")

# Region and window follow the starting example on slide 34 of 04 - Methods - SVM.pdf.
hydropathy_length = 40
hydropathy_window = 5


## 2. Feature extraction

We chose a starting set of 22 features from slides 30–34: 20 amino-acid fractions and two summaries of a hydropathy profile. Additional features are encouraged in the practical material, but charge, volume and other scales are optional extensions rather than assigned requirements here.

### 2.1 N-terminal amino-acid composition

**Implemented — Person 2.**

We defined `amino_acid_composition(sequence, n)`, which counts the canonical residues in the available first n positions and returns their fractions in the alphabet order `ACDEFGHIKLMNPQRSTVWY`. The 20 values are named `aa_fraction_A` to `aa_fraction_Y` and sum to 1 (slide 30).

Following the residue policy in 1.2, non-canonical residues are excluded from both the counts and the denominator, so each fraction describes the observed canonical residues rather than the full segment. Two training proteins (Q27972 and P02809) carry an X in this region; the remaining 8,144 are fully canonical, and no benchmarking protein is affected. A segment containing no canonical residue returns missing values rather than zeros, so an empty denominator cannot be reported as a composition of zeros.

The function receives n from its caller and never estimates it. It does not read `cleavage_pos`, so a protein's own annotation cannot choose its region; only `get_n` uses that column, on training positives.

We checked the function against the worked example on slide 30. For the first 21 residues of `MRFLAATFLLLALSTAAQAEPQF` it reproduces the published vector: A 0.29, L 0.24, F 0.10, T 0.10, and 0.05 for E, M, P, Q, R and S. For display, we listed all 20 amino acids in one table, including zero fractions and using two decimal places. The extractor's column order stays unchanged.

**Similar code:** the practical notebooks do not include this composition function. We adapted the canonical counting in 02c section 5.2 from pooled class counts to individual proteins; slide 30 defines the required vector.

In [15]:
composition_feature_names = [f"aa_fraction_{amino_acid}" for amino_acid in aa_alphabet]


# This function adapts the counting in section 5.2 of 02c_data_analysis.ipynb
# to individual proteins, producing the vector defined on slide 30 of the SVM slides.
def amino_acid_composition(sequence, n):
    """Fractions of the canonical residues within the first n positions."""
    counts = dict.fromkeys(aa_alphabet, 0)
    observed = 0
    for residue in sequence[:n]:
        if residue in counts:
            counts[residue] += 1
            observed += 1
    if observed == 0:
        return {name: np.nan for name in composition_feature_names}
    return {
        f"aa_fraction_{amino_acid}": counts[amino_acid] / observed
        for amino_acid in aa_alphabet
    }


# This sequence and n=21 reproduce the worked example on slide 30 of 04 - Methods - SVM.pdf.
slide_example = amino_acid_composition("MRFLAATFLLLALSTAAQAEPQF", 21)
composition_table = pd.DataFrame(
    [[slide_example[f"aa_fraction_{amino_acid}"] for amino_acid in aa_alphabet]],
    columns=aa_alphabet,
    index=["Fraction"],
)

print("N-terminal amino acid composition for slide example:")

with pd.option_context("display.max_columns", 20):
    display(composition_table.round(2))

print("Fractions sum to:", round(sum(slide_example.values()), 12))


N-terminal amino acid composition for slide example:


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Fraction,0.29,0.0,0.0,0.05,0.1,0.0,0.0,0.0,0.0,0.24,0.05,0.0,0.05,0.05,0.05,0.05,0.1,0.0,0.0,0.0


Fractions sum to: 1.0


### 2.2 Hydropathy profile and summaries

**Implemented — Person 2.**

We wrote the profile explicitly instead of calling `protein_scale`, because Biopython handles residues outside the scale by warning and skipping their symmetric partners within a window, whereas our policy in 1.2 averages the known values without shifting positions. `hydropathy_profile(sequence)` takes the first 40 residues, maps each to its Kyte–Doolittle value, and averages the known values inside every complete 5-position window with uniform weights and no padding. A canonical segment of length L therefore gives L − w + 1 window values; the shortest training protein is exactly 40 residues long, so every training profile contains 36 values. A window whose five positions are all unknown is recorded as missing.

`hydropathy_features(sequence)` summarizes the non-missing window values as `hydropathy_max_w5` and `hydropathy_profile_mean_w5` (slide 34). The mean is taken over the window profile rather than over the 40 residue values directly, so the first and last four residues contribute to fewer windows than the interior ones.

We compared the first 300 fully canonical N-termini with `ProteinAnalysis.protein_scale(kd, 5)` and obtained a largest difference of 8.9e-16. The saved output also shows that a sequence of 40 unknown residues returns missing values for both features instead of zero. Replacing a residue with X would preserve the profile length and could affect only windows containing that position; this follows from keeping the original positions when forming windows.

The maximum and the mean describe the same profile from two directions. The maximum reports the strongest local hydrophobic peak, which may reflect an SP h-region or another hydrophobic segment, and the mean reports how hydrophobic the N-terminus is overall; a short strong peak and a broad mild one can share a mean but not a maximum.

**Similar code:** `window_profile` in hydrophobicity notebook A.2 is the closest loop, and A.3 compares it with `ProteinAnalysis.protein_scale`. A.5 explains the unknown-residue issue, which we handled with the complete windows chosen in 1.2 rather than its shrinking windows; B.4 shows the profile-to-feature aggregation. Slide 34 gives the max/mean pair.

In [16]:
hydropathy_feature_names = [
    f"hydropathy_max_w{hydropathy_window}",
    f"hydropathy_profile_mean_w{hydropathy_window}",
]


# This function combines A.1 (residue values) and A.2 (window_profile) in
# 04_feature_extraction_hydrophobicity.ipynb, with the unknown-residue policy from A.5.
def hydropathy_profile(sequence):
    """Complete-window Kyte-Doolittle profile over the first hydropathy_length residues."""
    # Similar to A.1: look up one scale value per residue, retaining unknown positions.
    values = [
        hydropathy_scale.get(residue, np.nan)
        for residue in sequence[:hydropathy_length]
    ]
    profile = []
    # Similar to window_profile in A.2: slide only complete windows along the sequence.
    for start in range(len(values) - hydropathy_window + 1):
        # Adapted from A.5: average known residues, without shrinking the end windows.
        window_values = [
            value
            for value in values[start : start + hydropathy_window]
            if not np.isnan(value)
        ]
        profile.append(float(np.mean(window_values)) if window_values else np.nan)
    return profile


# This function follows profile aggregation in B.4 of
# 04_feature_extraction_hydrophobicity.ipynb and the max/mean pair on SVM slide 34.
def hydropathy_features(sequence):
    """Maximum and mean of the non-missing window values of the profile."""
    profile = [value for value in hydropathy_profile(sequence) if not np.isnan(value)]
    if not profile:
        return {name: np.nan for name in hydropathy_feature_names}
    return dict(zip(hydropathy_feature_names, [max(profile), float(np.mean(profile))]))


# This comparison follows A.3 in 04_feature_extraction_hydrophobicity.ipynb:
# compare our explicit loop with ProteinAnalysis.protein_scale on canonical sequences.
from Bio.SeqUtils.ProtParam import ProteinAnalysis

canonical_sequences = [
    sequence
    for sequence in data["sequence"]
    if set(sequence[:hydropathy_length]) <= set(aa_alphabet)
]
largest_difference = max(
    abs(ours - reference)
    for sequence in canonical_sequences[:300]
    for ours, reference in zip(
        hydropathy_profile(sequence),
        ProteinAnalysis(sequence[:hydropathy_length]).protein_scale(
            hydropathy_scale, hydropathy_window
        ),
    )
)

print(f"Canonical N-termini: {len(canonical_sequences)} of {len(data)} training proteins")
print(f"Largest difference from protein_scale (first 300): {largest_difference:.1e}")
print("Profile values per 40-residue segment:", len(hydropathy_profile("A" * 40)))
# Project check for the missing-value behavior motivated by A.5.
print("All-unknown segment:", hydropathy_features("X" * 40))

Canonical N-termini: 8144 of 8146 training proteins
Largest difference from protein_scale (first 300): 8.9e-16
Profile values per 40-residue segment: 36
All-unknown segment: {'hydropathy_max_w5': nan, 'hydropathy_profile_mean_w5': nan}


### 2.3 Assemble the feature extractor and catalog

**Implemented — Person 2.**

`extract_features(proteins, n)` applies the two extractors to every row of the supplied table and returns an accession-indexed DataFrame with the 20 composition columns followed by the two hydropathy columns. The column order comes from the shared `feature_names` list, so every training, validation and test subset is described in the same order; `D = 22` is the feature count Person 3 needs for the k grid (slide 38). Row order follows the caller's table, which keeps the matrix aligned with its label array.

Only the 22 predictors leave this function. Labels, folds, lengths, organism, `cleavage_pos` and `has_tm_in_90` stay in the metadata tables, so nothing derived from an annotation that would be unavailable for a new protein can reach the model.

Composition depends on the supplied n, while hydropathy uses the fixed region and window settings. Changing n from 22 to 30 could therefore change the composition values but would leave the hydropathy values unchanged. Person 4 could reuse the hydropathy block for each protein while recalculating composition with the n of the current training subset.

The catalog records, for each feature, its definition, the sequence region it uses, the scale, the window, the aggregation and the unit. It distinguishes the variable first-n composition region from the fixed 40-residue hydropathy region, which is the distinction Person 5 must preserve when exporting it in 6.2. Any feature added later needs a catalog row and an updated D.

The example extraction below shows the matrix size and the number of missing values. It uses the full training split only to demonstrate the extractor, without fitting a model. During CV, sections 4 and 5 will estimate n from each training subset and extract its corresponding features.

**Similar code:** hydrophobicity notebook B.4 assembles a small feature dictionary but does not build a protein matrix. Slide 36 shows how these features enter the model, and slide 38 ties D to the k grid. No global composition matrix is reused across CV splits.

In [17]:
# Project feature order; D is the feature count used for the grid on SVM slide 38.
feature_names = composition_feature_names + hydropathy_feature_names
D = len(feature_names)


# Project wrapper around the composition and profile extractors; no direct equivalent
# in the examples. B.4 motivates fixed-size features, and SVM slide 36 combines them.
def extract_features(proteins, n):
    """Composition over the first n residues plus hydropathy features, indexed by accession."""
    records = []
    for sequence in proteins["sequence"]:
        record = amino_acid_composition(sequence, n)
        record.update(hydropathy_features(sequence))
        records.append(record)
    features = pd.DataFrame(
        records, index=pd.Index(proteins["accession"], name="accession")
    )
    return features[feature_names]


# Project documentation table for our feature definitions:
feature_catalog = pd.DataFrame(
    [
        {
            "feature": f"aa_fraction_{amino_acid}",
            "definition": f"Fraction of the canonical residues that are {amino_acid}",
            "region": "first n residues (n estimated from the training positives)",
            "scale": "none (residue counts)",
            "window": pd.NA,
            "aggregation": "fraction",
            "unit": "fraction of canonical residues",
        }
        for amino_acid in aa_alphabet
    ]
    + [
        {
            "feature": hydropathy_feature_names[0],
            "definition": "Maximum of the Kyte-Doolittle window profile",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": "Kyte-Doolittle",
            "window": hydropathy_window,
            "aggregation": "maximum",
            "unit": "hydropathy index",
        },
        {
            "feature": hydropathy_feature_names[1],
            "definition": "Mean of the Kyte-Doolittle window profile",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": "Kyte-Doolittle",
            "window": hydropathy_window,
            "aggregation": "mean",
            "unit": "hydropathy index",
        },
    ]
)

# Example extraction: a project demonstration of the assembled matrix.
# CV will estimate n separately in each training subset, as on SVM slide 40.
example_features = extract_features(data, get_n(data))

print("Example extraction:", example_features.shape)
print("Missing values:", int(example_features.isna().sum().sum()))

feature_catalog


Example extraction: (8146, 22)
Missing values: 0


,feature,definition,region,scale,window,aggregation,unit
0,aa_fraction_A,Fraction of the canonical residues that are A,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
1,aa_fraction_C,Fraction of the canonical residues that are C,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
2,aa_fraction_D,Fraction of the canonical residues that are D,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
3,aa_fraction_E,Fraction of the canonical residues that are E,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
4,aa_fraction_F,Fraction of the canonical residues that are F,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
5,aa_fraction_G,Fraction of the canonical residues that are G,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
6,aa_fraction_H,Fraction of the canonical residues that are H,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
7,aa_fraction_I,Fraction of the canonical residues that are I,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
8,aa_fraction_K,Fraction of the canonical residues that are K,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
9,aa_fraction_L,Fraction of the canonical residues that are L,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues


## 3. RF selection and SVM helpers

We followed the order in slides 35–36: extract features, rank them with RF, keep top-k, standardize selected columns, and train an RBF SVM. Person 3 prepares the helpers; Person 4 will call them inside CV.

### 3.1 Fit an RF ranking

**Pending — Person 3; receives Person 2's extractor and feature order.** Define a helper that fits RF on the supplied training matrix and labels, returning ranked column indices, feature names and importances. Use fixed settings of 500 trees and `random_state = 0`, as in slide 35.

**Deliverable:** one ranking per current training set. Reuse that ranking for every candidate k, C and gamma. With the initial grid, the five outer rounds require 20 inner-training rankings and five outer-training rankings; do not fit a new forest for every parameter tuple.

**Similar code:** slide 35 provides the fit and ranking operations. The theoretical worked example has `select_rf`, but refits a forest for each k; use slide 35's reuse strategy for this practical.

### 3.2 Build the coupled parameter grid

**Pending — Person 3.** Use D from the extractor's actual feature count. Build unique valid k values from ceiling(D/4), ceiling(D/2) and D; with the starting D = 22, these are 6, 11 and 22. Use C values 1, 4 and 16. For each k, use gamma values 0.5/k, 1/k and 2/k.

**Deliverable:** 27 initial `(k, C, gamma)` tuples for the starting feature set, in a documented deterministic order. Generate gamma for each k; it is not one independent numerical gamma list shared by every k. Define exact ties to retain the first tuple, with smaller k considered first.

**Similar code:** slide 38 gives the grid; slide 40 shows its use. `04_SVM-sklearn_revised.ipynb` explains gamma in sections 4.2 and 6 but does not implement this joint search.

### 3.3 Standardize selected features and fit SVC

**Pending — Person 3.** Define helpers receiving selected training columns, training labels and a candidate C/gamma. Fit `StandardScaler` on those training columns and fit an RBF SVC on the transformed values. Prediction must apply the same fitted scaler to validation/test columns in the same order.

**Deliverable:** a fitted scaler/SVC pair, or a simple scaler/SVC Pipeline, for Person 4's candidate evaluation and outer refit. Leave probability calibration and additional model settings outside this practical workflow.

**Similar code:** SVM notebook section 3 provides scaling and SVC fitting; section 7 shows a Pipeline keeping the scaler and SVC together. Its fixed parameters and random dataset split are demonstrations; use the candidates and saved-fold subsets from this guide.

### 3.4 Calculate classification metrics

**Pending — Person 3.** Prepare a helper receiving true labels and predictions, returning MCC, precision, recall, F1 and accuracy with the same names as 03. Positive class 1 means SP. Use MCC alone to score candidates; the full metric set will describe external-fold performance.

**Deliverable to Person 4:** the ranking, grid, fitting/prediction and metric helpers. The next block receives definitions rather than a model trained on all proteins.

**Similar code:** `evaluate` in SVM notebook section 1 calculates these five metrics. Adapt its printed output to a result record; 03 section 5 supplies the project's table naming conventions.

## 4. Inner model selection

We chose explicit loops from slide 40 so the changing composition length and reused RF ranking are visible. Selection will use the four saved folds available within each outer-training set, with three for training and one for validation.

### 4.1 Select outer and inner subsets

**Pending — Person 4; receives the completed helpers from Person 3.** Define the five outer rounds from saved `cv_fold` identifiers 1–5. For each round, hold one fold out as test and use the other four as outer training. Within those four, choose each in turn as inner validation and use the other three for inner training.

**Deliverable:** subset selection for the nested loops. The slides label subsets 0–4; our existing 1–5 labels identify the same five-fold structure and remain unchanged. Benchmarking is excluded.

**Similar code:** slides 37 and 40 describe these subsets. Neither practical notebook implements the full nested protocol. The cyclic `get_cv_split` from 03 needs replacement because it uses one validation fold per outer round.

### 4.2 Prepare each inner-training fit

**Pending — Person 4.** For each inner split, compute n from its three-fold training positives using `get_n`. Call `extract_features` on inner training and validation with that same n. Fit one RF ranking using inner-training features and labels.

**Deliverable:** training/validation matrices, aligned labels, n and the ranking for the current inner split. These can be retained within the current outer round while scoring the grid. The next inner split must estimate its own n and ranking.

**Similar code:** slide 40 gives this exact sequence. The feature functions from the hydrophobicity notebook support extraction, but the training-dependent composition length is our adaptation to the practical instructions.

### 4.3 Score the grid and choose a combination

**Pending — Person 4.** For each `(k, C, gamma)` candidate, take the first k ranked columns, fit the scaler/SVM on inner training, predict inner validation and record MCC in `inner_scores`. Average each candidate's MCC across the four validation folds and choose the highest mean, using the tie convention in 3.2.

Before the outer refit, inspect whether the optimum lies on a searchable grid boundary. Extend the relevant candidate range using only inner-validation results, as required by slide 38, and record the final grid used. k cannot exceed D; selecting all D columns is a dimensional limit. Reuse the appropriate inner-training matrices and rankings when adding candidates.

**Deliverable:** best k/C/gamma for the current outer round and the candidate scores. Inner maxima are selection scores, not performance estimates. All choices must be complete before predicting the outer test fold.

**Similar code:** slides 38–40 are the primary reference. The practical SVM notebook does not include this search; it deliberately fixes parameters to explain SVC behavior.

## 5. Outer refit and evaluation

We planned one external test per saved fold, after all inner choices have been made. The procedure will be repeated five times, without repartitioning the data (slides 37–41).

### 5.1 Refit on the four outer-training folds

**Pending — Person 4.** Recompute n using all positive proteins in the current four-fold outer-training set. Extract outer-training and outer-test features with that n. Fit a new RF ranking on outer training, select its best k columns, and refit the scaler/SVM with the chosen C and gamma.

**Deliverable:** the final fitted procedure for this round, its n, selected feature names and their ranks/importances. Do not reuse one inner split's n, ranking or model as the final refit.

**Similar code:** the refit block in slide 40 specifies every step. SVM notebook section 3 supports scaler/SVC fitting, and hydrophobicity notebook A/B.4 supports profile extraction and aggregation.

### 5.2 Predict and record all five external folds

**Pending — Person 4.** Predict the held-out outer test proteins once with the refitted procedure and calculate the five test metrics. Repeat sections 4–5 for all saved test folds. Preserve accessions and labels in `cv_predictions`, and record n, selected k/C/gamma and metrics in `cv_per_round`. Record the outer-refit selection in `selected_features`.

**Deliverable to Person 5:** five result rows, test predictions for all 8,146 training proteins once each, selected-feature records and `inner_scores`. These counts describe the intended coverage, not executed results. Do not tune against external predictions or evaluate benchmarking here.

**Similar code:** slides 39–41 define the outputs. SVM notebook section 1 provides `evaluate`; its single holdout demonstration is not the five-round project protocol.

## 6. Results and integration

We followed 03's reporting conventions and the practical requirement to report each metric as mean ± standard error across the five external folds. All tables, figures and observations below are pending.

### 6.1 Per-round results and mean ± SE

**Pending — Person 5; receives Person 4's completed CV results.** Display `cv_per_round` and summarize precision, recall, accuracy, MCC and F1 across the five rounds. Use the sample standard deviation (`ddof = 1`) and SE = SD / sqrt(5), matching 03. Keep summary columns `mean`, `std` and `stderr`, indexed by metric.

**Deliverable:** `table_dir / 6_1_cv_per_round.tsv`, `table_dir / 6_1_cv_summary.tsv` and `table_dir / 6_1_cv_predictions.tsv`. Save full precision and round only for display; label reported uncertainty as SE.

**Similar code:** 03 sections 5.4–5.5 already save per-round tables and calculate mean ± SE. The practical SVM notebook section 1 supplies individual metrics; slides 39–41 require the five-round summary.

### 6.2 Selected settings and feature records

**Pending — Person 5.** Show the selected n, k, C and gamma for each outer round, together with feature names and their outer-training importances. Export `selected_features`, `inner_scores` and the feature catalog. Describe variation across training sets without treating one ranked list as a biological finding.

**Deliverable:** `table_dir / 6_2_selected_features.tsv`, `table_dir / 6_2_inner_scores.tsv` and `table_dir / 6_2_feature_catalog.tsv`. The catalog must distinguish variable first-n composition from the fixed 40-residue hydropathy segment.

**Similar code:** slide 39 illustrates the settings table; slide 35 exposes importances. The two practical notebooks do not export these project tables. Follow the TSV convention in 03 section 5.

### 6.3 Plot external-fold metrics

**Pending — Person 5.** Plot the five metric means with SE and individual external-fold values, using consistent labels and colors. Save the figure as `figure_dir / 6_3_cv_metrics.pdf`, print its path and display it.

**Deliverable:** the external-fold metric figure. The points and error bars must come from the result tables; do not use illustrative values from slide 39 or outputs from either example dataset.

**Similar code:** 03 section 5.6 provides the mean ± SE plot. The practical notebooks show other plots, but not this five-round summary.

### 6.4 Observations and final integration

**Pending — Person 5.** Write brief observations about SP detection, parameter variation and selected features, using the actual external results. Read `../results/tables/03/5_4_cv_per_round.tsv` and `../results/tables/03/5_5_cv_summary.tsv` only for a descriptive comparison with the completed Von Heijne method; its training protocol differs from the SVM protocol.

Bring the five contributions together, retain the scientific order, and replace completed task instructions with explanations of what we actually did. Update README status/results when supported by real implementation and checks. Keep benchmarking reserved, and leave any unfinished tasks marked pending.

**Deliverable:** the integrated notebook and a short results-based discussion. Model internals, probability calibration, repeated random splits and all toy exercises are supporting examples rather than extra assigned analyses.

**Similar code:** the interpretation in 03 section 5.10 is a style reference. The practical examples explain their own proteins/datasets, so their observations cannot serve as our results.